## https://hf-mirror.com/docs/peft/task_guides/ia3

In [1]:
# A100 train
import os
		
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [2]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files="./ChnSentiCorp_htl_all.csv", split="train")
dataset = dataset.filter(lambda x: x["review"] is not None)
dataset

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 7766 examples [00:00, 93673.15 examples/s]
Filter: 100%|██████████| 7766/7766 [00:00<00:00, 281570.89 examples/s]


Dataset({
    features: ['label', 'review'],
    num_rows: 7765
})

In [3]:
ds = dataset.train_test_split(test_size=0.2)
ds

DatasetDict({
    train: Dataset({
        features: ['label', 'review'],
        num_rows: 6212
    })
    test: Dataset({
        features: ['label', 'review'],
        num_rows: 1553
    })
})

In [4]:
ds['train'][0]

{'label': 0,
 'review': '怎么说呢。以北京这种地方的房价以及房间质量来说。这价格已经算便宜的了。因为先前住的几个北京的宾馆，都是又贵服务又差而且房间相当小。平安府的房间也不大。特别是厕所，太狭小了，房间窗户对着墙壁和暖水设备，根本开不了窗太吵了，也晒不进太阳，而且宾馆里没有电梯的，得自己提着拉箱抬上三楼房间，中间服务人员也没人来帮忙。去结帐时，前台送了张VIP卡，说下次您来直接给我们打电话预定，我们的价要比携程要便宜！'}

In [5]:
from transformers import AutoTokenizer

text_column = "review"
label_column = "label"
max_length = 128

tokenizer = AutoTokenizer.from_pretrained("bigscience/mt0-large")

In [6]:
def preprocess_function(examples):
    inputs = examples['review']
    targets = [str(label) for label in examples['label']]
    model_inputs = tokenizer(inputs, max_length=max_length, padding="max_length", truncation=True, return_tensors="pt")
    labels = tokenizer(targets, max_length=3, padding="max_length", truncation=True, return_tensors="pt")
    labels = labels["input_ids"]
    labels[labels == tokenizer.pad_token_id] = -100
    model_inputs["labels"] = labels
    return model_inputs

In [7]:
processed_ds = ds.map(
    preprocess_function,
    batched=True,
    remove_columns=ds["train"].column_names
)

Map: 100%|██████████| 1553/1553 [00:00<00:00, 8533.93 examples/s]


In [8]:
print(processed_ds['train'][0])

{'input_ids': [259, 7427, 5435, 29518, 306, 4002, 8467, 39069, 25793, 493, 192824, 19300, 109277, 48219, 58524, 306, 5144, 21919, 17909, 13062, 114715, 493, 1322, 306, 22131, 6668, 2884, 15543, 493, 83656, 8467, 493, 186356, 261, 24134, 9785, 31568, 9893, 9785, 15915, 39722, 109277, 48965, 2144, 306, 118468, 43974, 493, 109277, 71268, 1146, 306, 132303, 162946, 4398, 261, 8227, 109225, 2144, 1322, 261, 109277, 68651, 27841, 2991, 3913, 71922, 25403, 1107, 51777, 2146, 24520, 261, 70506, 5028, 63171, 68651, 8227, 220259, 1322, 261, 3722, 112106, 1597, 10537, 103859, 261, 39722, 186356, 5811, 13342, 173958, 493, 261, 5880, 9512, 17459, 3913, 9352, 15471, 172089, 1644, 2092, 23413, 109277, 261, 162981, 9893, 27095, 200396, 1193, 3480, 46912, 59800, 306, 6072, 22312, 133982, 3582, 261, 2884, 4457, 7154, 1322, 8922, 30085, 11269, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [9]:
from torch.utils.data import DataLoader
from transformers import default_data_collator

train_ds = processed_ds["train"]
eval_ds = processed_ds["test"]

batch_size = 32

train_dataloader = DataLoader(
    train_ds, shuffle=True, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True
)
eval_dataloader = DataLoader(eval_ds, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True)

In [10]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained("bigscience/mt0-large")

In [11]:
from peft import IA3Config, get_peft_model

peft_config = IA3Config(task_type="SEQ_2_SEQ_LM")
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 282,624 || all params: 1,229,863,936 || trainable%: 0.0230


In [12]:
import torch
from transformers import get_linear_schedule_with_warmup

lr = 8e-3
num_epochs = 3

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=(len(train_dataloader) * num_epochs),
)

In [13]:
from tqdm import tqdm

device = torch.accelerator.current_accelerator().type if hasattr(torch, "accelerator") else "cuda"
model = model.to(device)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for step, batch in enumerate(tqdm(train_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.detach().float()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

    model.eval()
    eval_loss = 0
    eval_preds = []
    for step, batch in enumerate(tqdm(eval_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        loss = outputs.loss
        eval_loss += loss.detach().float()
        eval_preds.extend(
            tokenizer.batch_decode(torch.argmax(outputs.logits, -1).detach().cpu().numpy(), skip_special_tokens=True)
        )

    eval_epoch_loss = eval_loss / len(eval_dataloader)
    eval_ppl = torch.exp(eval_epoch_loss)
    train_epoch_loss = total_loss / len(train_dataloader)
    train_ppl = torch.exp(train_epoch_loss)
    print(f"{epoch=}: {train_ppl=} {train_epoch_loss=} {eval_ppl=} {eval_epoch_loss=}")

100%|██████████| 49/49 [00:12<00:00,  3.93it/s]


epoch=0: train_ppl=tensor(1.4825, device='cuda:0') train_epoch_loss=tensor(0.3938, device='cuda:0') eval_ppl=tensor(1.1616, device='cuda:0') eval_epoch_loss=tensor(0.1498, device='cuda:0')


100%|██████████| 49/49 [00:12<00:00,  3.94it/s]


epoch=1: train_ppl=tensor(1.1544, device='cuda:0') train_epoch_loss=tensor(0.1436, device='cuda:0') eval_ppl=tensor(1.1455, device='cuda:0') eval_epoch_loss=tensor(0.1359, device='cuda:0')


100%|██████████| 49/49 [00:12<00:00,  3.94it/s]

epoch=2: train_ppl=tensor(1.1425, device='cuda:0') train_epoch_loss=tensor(0.1332, device='cuda:0') eval_ppl=tensor(1.1353, device='cuda:0') eval_epoch_loss=tensor(0.1269, device='cuda:0')


In [ ]:
model.save_pretrained('./model')
tokenizer.save_pretrained('./model')

In [24]:
ds["test"]['review'][1500], ds["test"]['label'][1500]

('饭店坐地起价！协程上面6月份就定好8月10号6日6个标准间，每个300，到了那里变成380了，我们坚决不肯多出钱，饭店就说是没有6个房间，只有4个了。这不是故意变相加价吗？我们坚决不肯就范，要求协程出面解决！！最后给了总统套房，由协程出多出来的部分。如果大家以后碰到这种事情千万不能妥协，要求协程出面解决！',
 0)

In [25]:
from peft import AutoPeftModelForSeq2SeqLM

device = torch.accelerator.current_accelerator().type if hasattr(torch, "accelerator") else "cuda"

model = model.to(device)
tokenizer = AutoTokenizer.from_pretrained("bigscience/mt0-large")

i = 1500
inputs = tokenizer(ds["test"]['review'][i], return_tensors="pt")
print(ds["test"]['review'][i])

饭店坐地起价！协程上面6月份就定好8月10号6日6个标准间，每个300，到了那里变成380了，我们坚决不肯多出钱，饭店就说是没有6个房间，只有4个了。这不是故意变相加价吗？我们坚决不肯就范，要求协程出面解决！！最后给了总统套房，由协程出多出来的部分。如果大家以后碰到这种事情千万不能妥协，要求协程出面解决！


In [26]:
with torch.no_grad():
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model.generate(input_ids=inputs["input_ids"], max_new_tokens=10)
    print(tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True))

['0']
